# S6E9: Does Breaking Ties help?

## Recovering the exact 0.94657 version

My previous update scored **0.94656**. I had used F.A.Nina's latest version 7 output, while Kaggle's best-score link points to **version 4**. This update fixes that version mismatch.

I downloaded the historical version 4 output directly, without recomputing its blend from changing upstream inputs. [Kaggle's version 4 page](https://www.kaggle.com/code/nina2025/ps-s6e9-a-simple-50-50-two-level-straight-blend?scriptVersionId=351902031) explicitly links **0.94657** to this version (script version ID `351902031`). I checked the downloaded file against the historical output preview and pinned its SHA-256 fingerprint.

| Submission | Public LB | Evidence |
|---|---:|---|
| My previous version 7 reference | 0.94656 | I confirmed this submission score |
| F.A.Nina's historical version 4 | **0.94657** | Kaggle's best-score link identifies version 4 |
| My submission from this notebook | Pending | I still need to submit this exact file |

I reuse F.A.Nina's public output with attribution; I do not claim this blend as my own model. My title stays the same, but blending changes rankings, not only ties. This public-score result does not establish a private-leaderboard improvement.


In [1]:
from pathlib import Path
import csv
import hashlib
import json
import math
import shutil
from IPython.display import FileLink, display

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def find_input(filename):
    matches = sorted(INPUT_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(f'Attach the input containing {filename}, then run again.')
    if len(matches) > 1:
        hashes = {hashlib.sha256(path.read_bytes()).hexdigest() for path in matches}
        if len(hashes) != 1:
            raise ValueError(f'Multiple different copies of {filename}: remove the extra input.')
    return matches[0]

source_path = find_input('nina_v4_submission.csv')
sample_path = find_input('sample_submission.csv')
print('Prediction input:', source_path)


Prediction input: /kaggle/input/datasets/chinzorigtganbat/s6e9-lexsort-trial/nina_v4_submission.csv


## 2. Why version 4 is different

I inspected the historical version 4 notebook. Its calculation ends after two stages:

```text
first      = 0.50 × jazivxt + 0.50 × Nina h-blend
submission = 0.47 × first  + 0.54 × Nina rounded-reference output
```

Version 7 adds another 50/50 blend with a denpugovkin teacher-ensemble output. That changes rankings. My version 7 submission scored 0.94656; I am now testing the exact version 4 output linked to 0.94657.

I preserve the author's 0.54 coefficient and original exported scores. I do not normalize, clip, round, or add another blend. The displayed upstream version numbers alone are insufficient to reproduce a historical file reliably, so I use the archived final output.

Credits: [F.A.Nina's version 4 blend](https://www.kaggle.com/code/nina2025/ps-s6e9-a-simple-50-50-two-level-straight-blend?scriptVersionId=351902031), [jazivxt](https://www.kaggle.com/code/jazivxt/single-model-zoom-zoom), [Nina h-blend](https://www.kaggle.com/code/nina2025/ps-s6e9-h-blend-pandas-division-attention?scriptVersionId=351557817), and [Nina rounded-reference notebook](https://www.kaggle.com/code/nina2025/ps-s6e9-top-subm-will-buy-ev-round-5).


In [2]:
EXPECTED_SHA256 = 'b625edbaa8a715b3fd69d90fe1f23c9260bdd9cf95b975e04448449b09ab928e'
actual_hash = hashlib.sha256(source_path.read_bytes()).hexdigest()
assert actual_hash == EXPECTED_SHA256, 'Prediction file does not match the pinned version.'

def read_csv(path):
    with path.open(newline='', encoding='utf-8-sig') as handle:
        reader = csv.DictReader(handle)
        return reader.fieldnames, list(reader)

columns, predictions = read_csv(source_path)
sample_columns, sample = read_csv(sample_path)
assert columns == sample_columns == ['id', 'Will_Buy_EV'], 'Unexpected columns.'
assert len(predictions) == len(sample) == 286571, 'Unexpected number of rows.'
prediction_ids = [row['id'] for row in predictions]
assert prediction_ids == [row['id'] for row in sample], 'IDs or row order do not match.'
assert len(set(prediction_ids)) == len(prediction_ids), 'Duplicate IDs.'
scores = [float(row['Will_Buy_EV']) for row in predictions]
assert all(math.isfinite(score) for score in scores), 'Missing or non-finite scores.'
print(f'Validated {len(scores):,} rows. Score range: {min(scores):.8f} to {max(scores):.8f}')
print('Exact pinned source confirmed.')


Validated 286,571 rows. Score range: 0.00013560 to 0.98899402
Exact pinned source confirmed.


## 3. Export my submission

I copy the validated historical file unchanged to **`/kaggle/working/submission.csv`**. The final cell provides a download link. I can also save a notebook version and select this CSV in the competition submission flow.

My own result remains pending until I submit. The source's **0.94657** is now tied to **version 4**, rather than only to a notebook-level best score.


In [3]:
submission_path = OUTPUT_ROOT / 'submission.csv'
shutil.copyfile(source_path, submission_path)
assert hashlib.sha256(submission_path.read_bytes()).hexdigest() == EXPECTED_SHA256

record = {'source_url': 'https://www.kaggle.com/code/nina2025/ps-s6e9-a-simple-50-50-two-level-straight-blend?scriptVersionId=351902031', 'requested_version': 4, 'script_version_id': 351902031, 'output_request': {'ownerSlug': 'nina2025', 'kernelSlug': 'ps-s6e9-a-simple-50-50-two-level-straight-blend', 'versionNumber': 4, 'filePath': 'submission.csv'}, 'sha256': 'b625edbaa8a715b3fd69d90fe1f23c9260bdd9cf95b975e04448449b09ab928e', 'bytes': 7563958, 'author_reported_public_lb': 0.94657, 'my_public_lb': None, 'provenance': 'Kaggle best-score link explicitly identifies version 4. Download uses SDK versionNumber=4 and matches historical output preview.'}
record['rows'] = len(predictions)
record['output'] = 'submission.csv'
(OUTPUT_ROOT / 'submission_manifest.json').write_text(json.dumps(record, indent=2))
print('Ready:', submission_path)
display(FileLink(str(submission_path)))


Ready: /kaggle/working/submission.csv


/kaggle/working/submission.csv